In [15]:
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
import os

# Take environment variables from .env
load_dotenv(override=True)

# This notebook uses the following variables from your .env file
search_endpoint = os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT")
knowledge_base_name = os.getenv("KNOWLEDGE_BASE_NAME")

In [16]:
# The Knowledge Base exposes an MCP endpoint for tool-based access
mcp_endpoint = f"{search_endpoint}/knowledgebases/{knowledge_base_name}/mcp?api-version=2025-11-01-Preview"
print(f"✅ MCP endpoint: {mcp_endpoint}")
     

✅ MCP endpoint: https://rag-time.search.windows.net/knowledgebases/earth-knowledge-base/mcp?api-version=2025-11-01-Preview


In [18]:
import requests
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

credential = DefaultAzureCredential()

bearer_token_provider = get_bearer_token_provider(
    credential, "https://management.azure.com/.default"
)

headers = {
  "Authorization": f"Bearer {bearer_token_provider()}",
}

In [ ]:
import requests
from azure.identity import get_bearer_token_provider

FOUNDRY_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT = os.environ.get("FOUNDRY_MODEL_DEPLOYMENT_NAME")
PROJECT_RESOURCE_ID = os.environ["FOUNDRY_PROJECT_RESOURCE_ID"]
PROJECT_CONNECTION_NAME = "earth-kb-mcp-connection"


conn_response = requests.put(
    f"https://management.azure.com{PROJECT_RESOURCE_ID}/connections/{PROJECT_CONNECTION_NAME}?api-version=2025-10-01-preview",
    headers=headers,
    json={
        "name": PROJECT_CONNECTION_NAME,
        "type": "Microsoft.MachineLearningServices/workspaces/connections",
        "properties": {
            "authType": "ProjectManagedIdentity",
            "category": "RemoteTool",
            "target": mcp_endpoint,
            "isSharedToAll": True,
            "audience": "https://search.azure.com/",
            "metadata": {"ApiType": "Azure"},
        },
    },
)
conn_response.raise_for_status()
print(f"✅ Project connection '{PROJECT_CONNECTION_NAME}' created")

HTTPError: 400 Client Error: Missing discriminator property [AuthType] in request body. for url: https://management.azure.com/subscriptions/784e6c68-d702-4a8b-a678-2e0f2f36deab/resourceGroups/rg-chris-rag-time/providers/Microsoft.CognitiveServices/accounts/chris-rag-time-resource/projects/chris-rag-time/connections/earth-kb-mcp-connection?api-version=2025-10-01-preview

In [13]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool

project_client = AIProjectClient(
    endpoint=FOUNDRY_ENDPOINT, credential=credential
)

# Agent instructions optimized for knowledge retrieval with citations
instructions = """
You are a helpful assistant that must use the knowledge base to answer all the questions from user.
You must never answer from your own knowledge under any circumstances.
Every answer must always provide annotations for using the MCP knowledge base tool
and render them as: `【message_idx:search_idx†source_name】`
If you cannot find the answer in the provided knowledge base you must respond with "I don't know".
"""

# Create MCP tool with knowledge base connection
mcp_kb_tool = MCPTool(
    server_label="knowledge-base",
    server_url=mcp_endpoint,
    require_approval="never",
    allowed_tools=["knowledge_base_retrieve"],
    project_connection_id=PROJECT_CONNECTION_NAME,
)

agent = project_client.agents.create_version(
    agent_name="earth-at-night-agent",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT,
        instructions=instructions,
        tools=[mcp_kb_tool],
    ),
)

print(f"✅ Agent '{agent.name}' created (version={agent.version})")
     

✅ Agent 'earth-at-night-agent' created (version=1)


In [14]:
openai_client = project_client.get_openai_client()

# Create a conversation session
conversation = openai_client.conversations.create()

# Send request that triggers the MCP tool
response = openai_client.responses.create(
    conversation=conversation.id,
    input="Why is the Phoenix nighttime street grid so sharply visible from space?",
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
)

print("📝 Agent response:\n")
print(response.output_text)

BadRequestError: Error code: 400 - {'error': {'message': "Access denied when connecting to the MCP server at https://rag-time.search.windows.net:443/knowledgebases/earth-knowledge-base/mcp while enumerating tools (HTTP 403 Forbidden). Please verify: (1) your credentials have the necessary permissions to access this server, (2) any IP allowlists or network policies permit requests from this service, and (3) the server's access control configuration allows the requested operation.", 'type': 'invalid_request_error', 'param': None, 'code': 'tool_user_error'}}